<a href="https://colab.research.google.com/github/kousiknandy/pycolab/blob/main/flatten_nested_list.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import collections
import threading

class NestedIterator:
    def __init__(self, nestedlist, history_buffer_size=5):
        # Global state for iteration progression, shared by all threads
        self._stack = [iter(nestedlist)]
        self._global_lock = threading.Lock() # Lock to protect access to _stack
        self._history_buffer_size = history_buffer_size

        # Per-thread state for peeked item (now a deque) and history buffer
        self._thread_local_state = threading.local()

    def __iter__(self):
        return self

    # Helper to get the next raw item from the global iterator, protected by a lock
    def _get_next_raw_item(self):
        with self._global_lock:
            while self._stack:
                try:
                    item = next(self._stack[-1])
                    if isinstance(item, list):
                        self._stack.append(iter(item))
                        continue
                    return item
                except StopIteration:
                    self._stack.pop()
            raise StopIteration # End of all iterators

    def __next__(self):
        # Initialize thread-local peeked_item as deque if not present for this thread
        if not hasattr(self._thread_local_state, 'peeked_item'):
            self._thread_local_state.peeked_item = collections.deque()

        # Initialize thread-local history_buffer if not present for this thread
        if not hasattr(self._thread_local_state, 'history_buffer'):
            self._thread_local_state.history_buffer = collections.deque(maxlen=self._history_buffer_size)

        item_to_return = None

        # Access thread-local peeked_item (now a deque)
        if self._thread_local_state.peeked_item: # If deque is not empty
            item_to_return = self._thread_local_state.peeked_item.popleft() # Pop from the left
        else:
            # Get the next item from the global, synchronized iterator
            item_to_return = self._get_next_raw_item()

        # Add the item to *this thread's* history buffer
        self._thread_local_state.history_buffer.append(item_to_return)
        return item_to_return

    def peek(self):
        # Initialize thread-local peeked_item as deque if not present for this thread
        if not hasattr(self._thread_local_state, 'peeked_item'):
            self._thread_local_state.peeked_item = collections.deque()

        # Access thread-local peeked_item
        if not self._thread_local_state.peeked_item: # If deque is empty
            try:
                # Get the next item from the global, synchronized iterator
                # and push it to the left of the peeked_item deque
                new_item = self._get_next_raw_item()
                self._thread_local_state.peeked_item.appendleft(new_item)
            except StopIteration:
                return None
        return self._thread_local_state.peeked_item[0] # Return the leftmost item without removing it

    def prev(self, n=1):
        """
        Returns the n-th previous element from the history buffer of the current thread.
        Raises IndexError if n is out of bounds (n <= 0 or n > buffer length).
        """
        # Ensure the current thread has its own history buffer
        if not hasattr(self._thread_local_state, 'history_buffer'):
            raise IndexError("No history available for this thread yet. Call next() first to build history.")

        history_buffer = self._thread_local_state.history_buffer
        if not (0 < n <= len(history_buffer)):
            raise IndexError(f"Cannot go back {n} steps. History buffer has {len(history_buffer)} items for this thread. Please use a value between 1 and {len(history_buffer)}.")
        return history_buffer[-n]

    def push_back(self, item):
        """
        Pushes an item back to the front of the current thread's iteration stream.
        This item will be the next one returned by __next__ or peek().
        """
        if not hasattr(self._thread_local_state, 'peeked_item'):
            self._thread_local_state.peeked_item = collections.deque()
        self._thread_local_state.peeked_item.appendleft(item)


### Test Cases for NestedIterator

In [2]:
# Test Case 1: Simple nested list
list1 = [[1, 2], [3, 4]]
ni1 = NestedIterator(list1)
print(f"List 1: {list1}")
print(f"Flattened: {list(ni1)}\n")

# Test Case 2: Deeply nested list
list2 = [1, [2, [3, 4], 5], [6, 7]]
ni2 = NestedIterator(list2)
print(f"List 2: {list2}")
print(f"Flattened: {list(ni2)}\n")

# Test Case 3: List with empty nested lists
list3 = [1, [], [2, [], 3], []]
ni3 = NestedIterator(list3)
print(f"List 3: {list3}")
print(f"Flattened: {list(ni3)}\n")

# Test Case 4: Empty list
list4 = []
ni4 = NestedIterator(list4)
print(f"List 4: {list4}")
print(f"Flattened: {list(ni4)}\n")

# Test Case 5: List with only one level of nesting
list5 = [1, 2, 3, 4, 5]
ni5 = NestedIterator(list5)
print(f"List 5: {list5}")
print(f"Flattened: {list(ni5)}\n")

# Test Case 6: List with multiple empty lists
list6 = [[], [], [1,2], [], [3,4,5]]
ni6 = NestedIterator(list6)
print(f"List 6: {list6}")
print(f"Flattened: {list(ni6)}\n")

List 1: [[1, 2], [3, 4]]
Flattened: [1, 2, 3, 4]

List 2: [1, [2, [3, 4], 5], [6, 7]]
Flattened: [1, 2, 3, 4, 5, 6, 7]

List 3: [1, [], [2, [], 3], []]
Flattened: [1, 2, 3]

List 4: []
Flattened: []

List 5: [1, 2, 3, 4, 5]
Flattened: [1, 2, 3, 4, 5]

List 6: [[], [], [1, 2], [], [3, 4, 5]]
Flattened: [1, 2, 3, 4, 5]



### Test Case for `peek()` method

In [4]:
# Test Case 7: Using peek()
print("Test Case 7: Using peek()")
list7 = [1, [2, 3], 4, [5, [6]]]
ni7 = NestedIterator(list7)

print(f"List: {list7}")

# Demonstrate peek() followed by next()
print(f"Peeked: {ni7.peek()}") # Should be 1
print(f"Next: {next(ni7)}")    # Should also be 1

print(f"Peeked: {ni7.peek()}") # Should be 2
print(f"Next: {next(ni7)}")    # Should also be 2

print(f"Next: {next(ni7)}")    # Should be 3 (without peek first)

print(f"Peeked: {ni7.peek()}") # Should be 4
print(f"Next: {next(ni7)}")    # Should also be 4

print(f"Peeked: {ni7.peek()}") # Should be 5
print(f"Next: {next(ni7)}")    # Should also be 5

print(f"Peeked: {ni7.peek()}") # Should be 6
print(f"Next: {next(ni7)}")    # Should also be 6

try:
    ni7.peek()
except StopIteration:
    print("Reached end of iteration when trying to peek.")

try:
    next(ni7)
except StopIteration:
    print("Reached end of iteration when trying to get next.")

Test Case 7: Using peek()
List: [1, [2, 3], 4, [5, [6]]]
Peeked: 1
Next: 1
Peeked: 2
Next: 2
Next: 3
Peeked: 4
Next: 4
Peeked: 5
Next: 5
Peeked: 6
Next: 6
Reached end of iteration when trying to peek.
Reached end of iteration when trying to get next.


### Test Case for `prev()` method

In [8]:
# Test Case 8: Using prev()
print("Test Case 8: Using prev()")
list8 = [10, 20, [30, 40], 50, [60, 70, 80], 90, 100]
# Use a history buffer size larger than default to test various 'n' values
ni8 = NestedIterator(list8, history_buffer_size=10)

print(f"List: {list8}")

# Advance the iterator a few steps
for _ in range(5):
    print(f"Next: {next(ni8)}")

print(f"Previous 1st: {ni8.prev(1)}") # Should be 50
print(f"Previous 2nd: {ni8.prev(2)}") # Should be 40
print(f"Previous 3rd: {ni8.prev(3)}") # Should be 30

# Advance further
print(f"Next: {next(ni8)}") # Should be 60
print(f"Next: {next(ni8)}") # Should be 70

print(f"Previous 1st: {ni8.prev(1)}") # Should be 70
print(f"Previous 2nd: {ni8.prev(2)}") # Should be 60
print(f"Previous 3rd: {ni8.prev(3)}") # Should be 50

# Test edge cases for prev()
print("\nTesting prev() edge cases:")
try:
    ni8.prev(0)
except IndexError as e:
    print(f"Caught expected error: {e}")

try:
    ni8.prev(len(ni8._history_buffer) + 1)
except IndexError as e:
    print(f"Caught expected error: {e}")

# Test with a small buffer
list9 = [1, 2, 3, 4, 5]
ni9 = NestedIterator(list9, history_buffer_size=2)
print("\nTest with small history buffer (size 2):")
print(f"Next: {next(ni9)}") # 1
print(f"Next: {next(ni9)}") # 2
print(f"Next: {next(ni9)}") # 3 (1 should be dropped)
print(f"Previous 1st: {ni9.prev(1)}") # Should be 3
print(f"Previous 2nd: {ni9.prev(2)}") # Should be 2

try:
    ni9.prev(3)
except IndexError as e:
    print(f"Caught expected error with small buffer: {e}")


Test Case 8: Using prev()
List: [10, 20, [30, 40], 50, [60, 70, 80], 90, 100]
Next: 10
Next: 20
Next: 30
Next: 40
Next: 50
Previous 1st: 50
Previous 2nd: 40
Previous 3rd: 30
Next: 60
Next: 70
Previous 1st: 70
Previous 2nd: 60
Previous 3rd: 50

Testing prev() edge cases:
Caught expected error: Cannot go back 0 steps. History buffer has 7 items. Please use a value between 1 and 7.
Caught expected error: Cannot go back 8 steps. History buffer has 7 items. Please use a value between 1 and 7.

Test with small history buffer (size 2):
Next: 1
Next: 2
Next: 3
Previous 1st: 3
Previous 2nd: 2
Caught expected error with small buffer: Cannot go back 3 steps. History buffer has 2 items. Please use a value between 1 and 2.
